## RAG with Multi DataSource

In [ ]:
## Wikipedia Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper   

In [4]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
print(wiki.name)

wikipedia


In [9]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load data from the web
loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs = loader.load()

# 2. Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

# 3. Create Vectorstore with Ollama Embeddings
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# Using langchain-chroma's Chroma implementation
vectordb = Chroma.from_documents(
    documents=documents, 
    embedding=embeddings,
    collection_name="multi-agent-rag-chroma-mxbai-embed-large"
)

# 4. Prepare Retriever
retriever = vectordb.as_retriever()
print("Vector database created successfully!")

Vector database created successfully!


In [17]:
from langchain_core.tools import create_retriever_tool

retrieval_tools=create_retriever_tool(retriever,"langsmith_search","Search for information about Langsmith")

In [18]:
retrieval_tools.name

'langsmith_search'

In [20]:
# Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [21]:
tools=[wiki,arxiv,retrieval_tools]

In [22]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/manas/Desktop/Manas/Web/FastAPI/RAG with Multiple DataSources/.venv/lib/python3.11/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x114242f20>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x1113a1b20>)]

In [46]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1")
llm

ChatOllama(model='llama3.1')

In [47]:
## Prompt Template
from langchain_classic import hub

## get the prompt from hub
prompt = hub.pull("hwchase17/react")

# For llama2, it helps to be extremely strict about omitting conversational filler
prompt.template = "DO NOT use conversational filler (like 'Great, let\'s start!' or 'Please give me some time').\n" + prompt.template

print("ReAct prompt pulled and refined!")

ReAct prompt pulled and refined!


In [48]:
## Agents
from langchain_classic.agents import create_react_agent, AgentExecutor

agent = create_react_agent(llm, tools, prompt)

# Create the executor with a larger iteration limit and stricter parsing
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True,
    max_iterations=10
)
print("Agent successfully initialized!")

Agent successfully initialized!


In [ ]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...
Thought: To gather information about Langsmith, I need to determine which tool is most likely to have relevant information.
Action: langsmith_search
Action Input: query = 'Langsmith'LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agent